# Neural Network with PyTorch — Forward Propagation & Loss

This notebook is the **first part** of building a neural network from scratch with PyTorch, using the [Breast Cancer Wisconsin dataset](https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv).

Today we focus on two building blocks only:
1. **Forward propagation** — how the model turns input features into a predicted probability.
2. **Loss** — how we measure how wrong that prediction is.

We will **not** train the model yet (no backpropagation, no gradient updates). That means the weights stay random throughout this notebook — the goal here is purely to understand *what a forward pass computes* and *what the loss number means*. Training comes in the next class.

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

# Fix the random seed so weight initialization is reproducible across runs
torch.manual_seed(42)

### Load the data

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [ ]:
df.shape

(569, 33)

In [ ]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [ ]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


### train test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2, random_state=42)

### scaling

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
X_train

array([[-1.44075296, -0.43531947, -1.36208497, ...,  0.9320124 ,
         2.09724217,  1.88645014],
       [ 1.97409619,  1.73302577,  2.09167167, ...,  2.6989469 ,
         1.89116053,  2.49783848],
       [-1.39998202, -1.24962228, -1.34520926, ..., -0.97023893,
         0.59760192,  0.0578942 ],
       ...,
       [ 0.04880192, -0.55500086, -0.06512547, ..., -1.23903365,
        -0.70863864, -1.27145475],
       [-0.03896885,  0.10207345, -0.03137406, ...,  1.05001236,
         0.43432185,  1.21336207],
       [-0.54860557,  0.31327591, -0.60350155, ..., -0.61102866,
        -0.3345212 , -0.84628745]], shape=(455, 30))

In [ ]:
y_train

68     B
181    M
63     B
248    B
60     B
      ..
71     B
106    B
270    B
435    M
102    B
Name: diagnosis, Length: 455, dtype: object

### Label Encoding

In [ ]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
y_train

array([0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1,
       0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0,
       0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1,
       0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0,
       1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0,

### Numpy arrays to PyTorch tensors

In [ ]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [ ]:
X_train_tensor.shape

torch.Size([455, 30])

In [ ]:
y_train_tensor.shape

torch.Size([455])

### Defining the model

A single-layer neural network here does two things in its `forward` pass:

1. **Linear step** — combine all 30 input features into one number:
   $$z = X \cdot \text{weights} + \text{bias}$$
2. **Activation step** — squash that number into a probability between 0 and 1 using the sigmoid function:
   $$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

This `forward` pass alone doesn't learn anything — it just computes a prediction using whatever weights it currently has (right now: random values).

The `loss_function` then compares that prediction $\hat{y}$ to the true label $y$ using **binary cross-entropy**:
$$L = -\big(y \cdot \log(\hat{y}) + (1-y)\cdot \log(1-\hat{y})\big)$$

A high loss means the prediction was far from the true label; a low loss means it was close. Notice `requires_grad` is **not** set on the weights/bias here — we're not computing gradients or updating anything yet, so PyTorch's autograd tracking isn't needed this class.

In [ ]:
class MySimpleNN():

  def __init__(self, X):

    self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64)
    self.bias = torch.zeros(1, dtype=torch.float64)

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Calculate loss
    loss = -(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred)).mean()
    return loss

### Running a single forward pass

We create the model (which randomly initializes its weights and bias), run **one** forward pass on the training data, and compute the loss. We are not looping over epochs or updating any weights — this is just to see what a single forward pass + loss calculation looks like.

In [ ]:
# create model
model = MySimpleNN(X_train_tensor)

# forward pass (single pass, no training loop)
y_pred_train = model.forward(X_train_tensor)

# loss calculation
loss = model.loss_function(y_pred_train, y_train_tensor)

print(f'Predicted probabilities (first 5): {y_pred_train[:5].squeeze().tolist()}')
print(f'Loss with random (untrained) weights: {loss.item()}')

Predicted probabilities (first 5): [0.9999996210711913, 0.9999997788867343, 0.33022985027479096, 0.014736268738547357, 0.06706174662560076]
Loss with random (untrained) weights: 2.961569982845783


### Evaluation

Let's check the accuracy of this untrained model on the test set. Since the weights were never updated, they are still random — so we expect accuracy close to chance level (around 50% for a balanced binary classification problem).

This is the motivation for the next class: we need a way to *update* the weights so the loss goes down and accuracy goes up — that's what backpropagation and gradient descent will do.

In [ ]:
# model evaluation (weights are still random/untrained at this point)
with torch.no_grad():
  y_pred_test = model.forward(X_test_tensor)
  y_pred_labels = (y_pred_test > 0.5).float()
  accuracy = (y_pred_labels.squeeze() == y_test_tensor).float().mean()
  print(f'Accuracy with untrained (random) weights: {accuracy.item()}')

Accuracy with untrained (random) weights: 0.8947368264198303


### What's next

In the next class, we'll add:
- `requires_grad=True` on the weights and bias
- A training loop that calls `loss.backward()` to compute gradients (backpropagation)
- Gradient descent updates to actually reduce the loss over multiple epochs

By comparing the loss/accuracy here (random weights) to the loss/accuracy after training, you'll be able to see directly what training accomplishes.